**PHASE 4: ANALYSIS — Research Question 6**

Purpose: Compute publication year spread per cluster to classify research communities by temporal profile  
Input: arxiv_text_cleaned.pkl, cluster_labels_500d.pkl, cluster_profiles_500d.pkl, q4_growing_niches.pkl  
Output: q6_temporal_cohesion.pkl, results/q6_temporal_cohesion_summary.txt, temporal_cohesion.json  
Algorithm: Descriptive statistics on year distributions (IQR, std, emergence score)  
ML Involved: ✗ NO — pure analysis of existing clustering results  
Runtime: ~5 minutes  

Research Question 6:  
"What is the temporal profile of each research cluster — is it a young emerging niche or an established long-running field?"

Key metric: IQR of publication years per cluster
- Tight IQR (<=4 years)  → cluster papers are temporally concentrated = emerging niche
- Wide  IQR (>10 years)  → cluster spans decades = established discipline

Emergence score: fraction of cluster papers published from 2020 onwards
- High emergence (>=0.50) → majority of papers are recent
- Low  emergence (<0.20)  → most papers are older, field has long history

2x2 quadrant classification (crossing IQR with growth rate from Q4):
- rocket  — tight IQR + high growth  (e.g. LLMs: barely existed before 2022)
- niche   — tight IQR + low growth   (recent but not yet accelerating)
- classic — wide  IQR + high growth  (reinvigorated established field)
- stable  — wide  IQR + low growth   (mature, long-running discipline)

In [ ]:
# imports

import pandas as pd
import numpy as np
import pickle
import joblib
import json
import os
import sys
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sys.path.append('..')
from config import RANDOM_STATE

# thresholds — adjust here if you want to experiment
TIGHT_IQR_THRESHOLD  = 4     # years; below = temporally concentrated
WIDE_IQR_THRESHOLD   = 10    # years; above = long-running
HIGH_EMERGENCE_CUTOFF = 0.50  # fraction of papers from 2020+
RECENT_YEAR_CUTOFF   = 2020
HIGH_GROWTH_THRESHOLD = 1.0   # growth_rate multiplier from Q4 data; above = high growth

In [ ]:
# load data

print('Loading cluster labels...')
labels_df = pd.read_pickle('../data/processed/cluster_labels_500d.pkl')
print(f'✓ labels: {len(labels_df):,} papers')

print('Loading metadata...')
df_text = pd.read_pickle('../data/processed/arxiv_text_cleaned.pkl')
print(f'✓ metadata: {len(df_text):,} rows')

print('Loading cluster profiles (500d)...')
profiles = joblib.load('../data/processed/cluster_profiles_500d.pkl')
print(f'✓ profiles: {profiles["n_clusters"]} clusters')

print('Loading Q4 niche data (growth rates)...')
q4 = joblib.load('../data/processed/q4_growing_niches.pkl')
niches = q4['niches_500d']
print(f'✓ Q4 niches: {len(niches)} clusters')

# merge cluster labels into metadata
df = df_text.merge(labels_df, on='id', how='inner')
df = df[df['year'].between(2007, 2025)].copy()
print(f'✓ merged with valid years: {len(df):,} papers (2007–2025)')

In [ ]:
# compute temporal statistics per cluster

print('COMPUTING TEMPORAL SPREAD PER CLUSTER')
print('=' * 60)

results = []

for cid in sorted(df['cluster_id'].unique()):
    years = df[df['cluster_id'] == cid]['year'].values.astype(float)
    n = len(years)

    mean_yr   = float(np.mean(years))
    median_yr = float(np.median(years))
    std_yr    = float(np.std(years))
    q1        = float(np.percentile(years, 25))
    q3        = float(np.percentile(years, 75))
    iqr       = q3 - q1

    emergence_score = float((years >= RECENT_YEAR_CUTOFF).mean())

    # growth rate from Q4 (already computed)
    growth_rate = float(niches[cid]['growth_rate']) if cid in niches else 0.0

    # quadrant classification
    tight  = iqr <= TIGHT_IQR_THRESHOLD
    recent = emergence_score >= HIGH_EMERGENCE_CUTOFF
    fast   = growth_rate >= HIGH_GROWTH_THRESHOLD

    if tight and fast:   quadrant = 'rocket'
    elif tight and not fast: quadrant = 'niche'
    elif not tight and fast: quadrant = 'classic'
    else:                quadrant = 'stable'

    top_terms = [t[0] for t in profiles['top_terms'][cid][:5]]

    results.append({
        'cluster_id':      cid,
        'size':            n,
        'mean_year':       round(mean_yr, 1),
        'median_year':     round(median_yr, 1),
        'std_year':        round(std_yr, 2),
        'iqr':             round(iqr, 2),
        'q1_year':         round(q1, 1),
        'q3_year':         round(q3, 1),
        'emergence_score': round(emergence_score, 4),
        'growth_rate':     round(growth_rate, 4),
        'tight':           tight,
        'recent':          recent,
        'quadrant':        quadrant,
        'top_terms':       top_terms
    })

tc_df = pd.DataFrame(results)

print(f'Quadrant breakdown:')
print(tc_df['quadrant'].value_counts().to_string())
print()
print(f'IQR range: {tc_df["iqr"].min():.1f} – {tc_df["iqr"].max():.1f} years')
print(f'Emergence score range: {tc_df["emergence_score"].min():.3f} – {tc_df["emergence_score"].max():.3f}')

In [ ]:
# inspect each quadrant

print('CLUSTERS BY QUADRANT')
print('=' * 60)

quadrant_labels = {
    'rocket':  'ROCKET  — tight IQR + high growth (field that exploded recently)',
    'niche':   'NICHE   — tight IQR + low growth  (recent but not yet accelerating)',
    'classic': 'CLASSIC — wide IQR  + high growth (established field reinvigorated)',
    'stable':  'STABLE  — wide IQR  + low growth  (mature long-running discipline)'
}

for q, label in quadrant_labels.items():
    subset = tc_df[tc_df['quadrant'] == q].sort_values('emergence_score', ascending=False)
    print(f'\n{label} ({len(subset)} clusters):')
    for _, row in subset.iterrows():
        print(f'  C{int(row["cluster_id"]):02d} | IQR={row["iqr"]:.1f}yr | emerge={row["emergence_score"]*100:.0f}% | growth={row["growth_rate"]:.2f} | {" ".join(row["top_terms"][:4])}')

In [ ]:
# visualize: 2x2 scatter (growth rate vs IQR) + emergence score histogram

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Research Question 6: Temporal Cohesion per Cluster (500d)', fontsize=13)

quadrant_colors = {
    'rocket':  '#10b981',
    'niche':   '#3b82f6',
    'classic': '#f59e0b',
    'stable':  '#6b7280'
}

# --- left: growth rate vs IQR scatter (the 2x2 quadrant plot) ---
for q, color in quadrant_colors.items():
    mask = tc_df['quadrant'] == q
    subset = tc_df[mask]
    # cap growth_rate for readability (LLM cluster is extreme outlier)
    capped_growth = subset['growth_rate'].clip(upper=20)
    axes[0].scatter(
        subset['iqr'],
        capped_growth,
        c=color, label=q, alpha=0.8,
        s=subset['size'] / subset['size'].max() * 300 + 30
    )
    # label a few notable clusters
    for _, row in subset.nlargest(2, 'emergence_score').iterrows():
        axes[0].annotate(
            f'C{int(row["cluster_id"]):02d}',
            (row['iqr'], min(row['growth_rate'], 20)),
            fontsize=7, ha='center', va='bottom'
        )

axes[0].axvline(TIGHT_IQR_THRESHOLD, color='gray', linestyle='--', alpha=0.5, linewidth=1)
axes[0].axhline(HIGH_GROWTH_THRESHOLD, color='gray', linestyle='--', alpha=0.5, linewidth=1)
axes[0].set_xlabel('IQR of Publication Years (temporal spread)', fontsize=11)
axes[0].set_ylabel('Growth Rate (capped at 20×)', fontsize=11)
axes[0].set_title('2×2 Quadrant: Spread vs. Growth\n(dot size = cluster size)')
axes[0].legend(title='Quadrant', fontsize=9)

# --- right: emergence score distribution by quadrant ---
for q, color in quadrant_colors.items():
    subset = tc_df[tc_df['quadrant'] == q]
    axes[1].scatter(
        subset['emergence_score'],
        subset['iqr'],
        c=color, label=q, alpha=0.8, s=60
    )
axes[1].axvline(HIGH_EMERGENCE_CUTOFF, color='gray', linestyle='--', alpha=0.5,
                label=f'{int(HIGH_EMERGENCE_CUTOFF*100)}% cutoff')
axes[1].axhline(TIGHT_IQR_THRESHOLD, color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Emergence Score (fraction of papers from 2020+)', fontsize=11)
axes[1].set_ylabel('IQR of Publication Years', fontsize=11)
axes[1].set_title('Emergence Score vs. Temporal Spread')
axes[1].legend(title='Quadrant', fontsize=9)

plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/q6_temporal_cohesion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Figure saved: results/figures/q6_temporal_cohesion.png')

In [ ]:
# save pkl and text summary

print('SAVING Q6 RESULTS')
print('=' * 60)

q6_results = {
    'temporal_stats':    results,
    'thresholds': {
        'tight_iqr':       TIGHT_IQR_THRESHOLD,
        'wide_iqr':        WIDE_IQR_THRESHOLD,
        'high_emergence':  HIGH_EMERGENCE_CUTOFF,
        'recent_year':     RECENT_YEAR_CUTOFF,
        'high_growth':     HIGH_GROWTH_THRESHOLD
    },
    'analysis_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

joblib.dump(q6_results, '../data/processed/q6_temporal_cohesion.pkl')
print('✓ Saved pkl: data/processed/q6_temporal_cohesion.pkl')

summary_path = '../results/q6_temporal_cohesion_summary.txt'
with open(summary_path, 'w') as f:
    f.write('RESEARCH QUESTION 6: TEMPORAL COHESION PER CLUSTER\n')
    f.write('=' * 60 + '\n')
    f.write(f'Analysis date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n\n')
    f.write(f'Thresholds: tight_iqr<={TIGHT_IQR_THRESHOLD}yr | high_emergence>={HIGH_EMERGENCE_CUTOFF} | high_growth>={HIGH_GROWTH_THRESHOLD}\n\n')
    f.write('QUADRANT SUMMARY:\n')
    f.write(tc_df['quadrant'].value_counts().to_string() + '\n\n')
    for q, label in quadrant_labels.items():
        subset = tc_df[tc_df['quadrant'] == q].sort_values('emergence_score', ascending=False)
        f.write(f'{label}:\n')
        for _, row in subset.iterrows():
            f.write(f'  C{int(row["cluster_id"]):02d} | IQR={row["iqr"]:.1f} | emerge={row["emergence_score"]*100:.0f}% | growth={row["growth_rate"]:.3f} | {" ".join(row["top_terms"][:4])}\n')
        f.write('\n')
    f.write('\nFULL TABLE:\n')
    f.write(tc_df[['cluster_id','quadrant','iqr','emergence_score','growth_rate','median_year','size']].sort_values('emergence_score', ascending=False).to_string(index=False))
print(f'✓ Saved summary: {summary_path}')

In [ ]:
# export temporal_cohesion.json for website
# also merges temporalCohesion field into clusterexploration.json

print('EXPORTING temporal_cohesion.json')
print('=' * 60)

def to_native(v):
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.floating,)): return float(v)
    if isinstance(v, np.ndarray): return v.tolist()
    if isinstance(v, bool): return bool(v)
    return v

clusters_json = [
    {
        'clusterId':      to_native(row['cluster_id']),
        'meanYear':       to_native(row['mean_year']),
        'medianYear':     to_native(row['median_year']),
        'stdYear':        to_native(row['std_year']),
        'iqr':            to_native(row['iqr']),
        'q1Year':         to_native(row['q1_year']),
        'q3Year':         to_native(row['q3_year']),
        'emergenceScore': to_native(row['emergence_score']),
        'quadrant':       row['quadrant'],
        'topTerms':       row['top_terms']
    }
    for _, row in tc_df.iterrows()
]

quadrant_counts = tc_df['quadrant'].value_counts().to_dict()

output = {
    'summary': {
        'thresholds': {
            'tightIqr':       TIGHT_IQR_THRESHOLD,
            'highEmergence':  HIGH_EMERGENCE_CUTOFF,
            'recentYearCutoff': RECENT_YEAR_CUTOFF
        },
        'quadrantCounts': {k: to_native(v) for k, v in quadrant_counts.items()}
    },
    'clusters': clusters_json
}

website_data = os.path.join('..', '..', 'arxiv-trends-website', 'src', 'data')
if os.path.exists(website_data):
    json_path = os.path.join(website_data, 'temporal_cohesion.json')
else:
    json_path = '../results/temporal_cohesion.json'
    print('  (website path not found — saving to results/ instead)')

with open(json_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'✓ Saved JSON: {json_path}')

# merge temporalCohesion field into clusterexploration.json
ce_path = os.path.join(website_data, 'clusterexploration.json') if os.path.exists(website_data) else None
if ce_path and os.path.exists(ce_path):
    with open(ce_path) as f:
        ce = json.load(f)
    tc_by_id = {r['clusterId']: r for r in clusters_json}
    updated = 0
    for cluster in ce['clusters']:
        if cluster['id'] in tc_by_id:
            cluster['temporalCohesion'] = tc_by_id[cluster['id']]
            updated += 1
    with open(ce_path, 'w') as f:
        json.dump(ce, f, indent=2)
    print(f'✓ Merged temporalCohesion into clusterexploration.json ({updated} clusters updated)')
else:
    print('  (clusterexploration.json not found — merge step skipped)')

print(f'\nQuadrant counts: {quadrant_counts}')

In [ ]:
# verify all Q6 outputs

print('Q6 OUTPUT VERIFICATION')
print('=' * 60)

q6_files = [
    '../data/processed/q6_temporal_cohesion.pkl',
    '../results/q6_temporal_cohesion_summary.txt',
    '../results/figures/q6_temporal_cohesion.png',
]

all_good = True
for filepath in q6_files:
    if os.path.exists(filepath):
        print(f'✓ {filepath} ({os.path.getsize(filepath)/1024:.1f} KB)')
    else:
        print(f'x MISSING: {filepath}')
        all_good = False

for jp in [os.path.join(website_data, 'temporal_cohesion.json'), '../results/temporal_cohesion.json']:
    if os.path.exists(jp):
        print(f'✓ {jp} ({os.path.getsize(jp)/1024:.1f} KB)')
        break

print()
if all_good:
    print('✓ ALL Q6 OUTPUTS VERIFIED — ready for website integration')
    print(f'  Next: build website section showing 2x2 quadrant scatter in Q4 tab')
    print(f'  and year histogram in Explore cluster detail modal')
else:
    print('x Some outputs missing — re-run cells above')